# Fate replacement, zeroing and trained masking

**Role:** Analysis.

Compare three reference choices for removing exclusive source identity: zero-to-unassigned, donor-weighted fate replacement, and a trained missing-fate indicator. The center cell and graph remain unchanged. Evaluate all eligible recipient/source combinations at observed size and on fixed-neighborhood N sweeps, with fixed or N-dependent donor weights fitted from training organoids only. Support filtering, geometric normalization and within-model paired contrasts remain explicit. An optional final section retains the matched full-versus-exclusive comparison. This workflow requires the size-conditioned/ masking run formats; generic marker zeroing is in the neighboring notebook.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Replacement figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from src.analysis.interventions.replacement import METHODS

REPLACEMENT_LABELS = {
    'zero': 'Zero → unassigned',  # Figure label for zero.
    'replacement_fixed': 'Replacement: fixed weights',  # Figure label for replacement_fixed.
    'replacement_size_dependent': 'Replacement: N-dependent weights',  # Figure label for replacement_size_dependent.
    'masking': 'Hide fate (trained mask)',  # Figure label for masking.
}  # Display labels for figure groups or model variants.
REPLACEMENT_COLORS = dict(zip(METHODS, ['#777777', '#2878b5', '#d45a31']))  # Display colors assigned to groups or model variants.

REPLACEMENT_COLORS['masking'] = '#7b3faa'

REPLACEMENT_METRICS = {
    'delta_relative': 'Δ curvature / (4π / Aref(N))',  # Axis label for the delta_relative effect scale.
    'delta_mu': 'Δ curvature',  # Axis label for the delta_mu effect scale.
    'delta_z': 'Δ transformed curvature',  # Axis label for the delta_z effect scale.
}  # Quantities included in the study summaries.

def replacement_plot_pair_grid(result, *, analysis='sweep', hop=1, metric='delta_relative', min_organoids=5):
    """Rows = unchanged recipient; columns = original source identity.

    Every panel uses the same cases for all three methods. Sweep cohorts also
    stay fixed across N. Panel y scales differ to preserve visible curve shape.
    """
    names = result['config']['markers']
    summary = result['summary']
    methods = result.get('methods', METHODS)
    k = len(names)
    fig, axes = plt.subplots(k, k, figsize=(2.8*k, 2.25*k), squeeze=False, sharex=True)
    for i, center in enumerate(names):
        for j, source in enumerate(names):
            ax = axes[i, j]
            ax.axhline(0, color='.8', lw=.7)
            if not summary.empty:
                frame = summary[(summary.analysis == analysis) & (summary.hop == hop) &
                    (summary.metric == metric) & (summary.center_marker == center) &
                    (summary.source_marker_name == source)].sort_values('n')
                for method in methods:
                    part = frame[frame.method == method].copy()
                    ok = part.n_organoids >= min_organoids
                    # Keep NaN gaps so unsupported bins are not connected across.
                    y = part['mean'].where(ok)
                    ax.plot(part.n, y, color=REPLACEMENT_COLORS[method], lw=1.4,
                            ls='--' if method == 'replacement_size_dependent' else '-', marker='.', ms=3)
                    ax.fill_between(part.n, part.ci_low.where(ok), part.ci_high.where(ok),
                                    color=REPLACEMENT_COLORS[method], alpha=.10)
                if frame.empty or not (frame.n_organoids >= min_organoids).any():
                    ax.text(.5, .5, 'Insufficient support', ha='center', transform=ax.transAxes, fontsize=8)
                else:
                    counts = frame.n_organoids
                    ax.text(.03, .96, f'org: {counts.min()}–{counts.max()}', va='top',
                            transform=ax.transAxes, fontsize=7, color='.4')
            else:
                ax.text(.5, .5, 'Insufficient support', ha='center', transform=ax.transAxes, fontsize=8)
            if i == 0:
                ax.set_title(f'Source: {source}', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'Center: {center}', fontsize=10)
            if i == k - 1:
                ax.set_xlabel('Supplied N' if analysis == 'sweep' else 'Observed N (bin median)')
            ax.set_xscale('log'); ax.tick_params(labelsize=7)
    handles = [Line2D([0], [0], color=REPLACEMENT_COLORS[m], label=REPLACEMENT_LABELS[m],
                      ls='--' if m == 'replacement_size_dependent' else '-') for m in methods]
    fig.legend(handles=handles, loc='upper center', ncol=len(methods), bbox_to_anchor=(.5, .978))
    mode = 'Fixed neighborhoods, size sweep' if analysis == 'sweep' else 'Static: observed size'
    model_label = result.get('model_label', 'Original exclusive FiLM')
    fig.suptitle(f'{model_label}: {mode} • exact hop {hop}\n'
                 f'{REPLACEMENT_METRICS[metric]} = intervened − intact prediction', y=1.01)
    fig.text(.5, .003, 'Unassigned-source zeroing is a no-op. Per-panel y scales. '
             'Shading: organoid bootstrap, conditional on fitted models and replacement references.', ha='center', fontsize=10)
    fig.tight_layout(rect=[0, .02, 1, .955])
    return fig

def replacement_plot_support(result):
    names = result['config']['markers']; support = result['support']
    fig, axes = plt.subplots(2, 2, figsize=(16, 13), squeeze=False)
    for row, analysis in enumerate(['observed', 'sweep']):
        for col, hop in enumerate([1, 2]):
            ax = axes[row, col]
            part = support[(support.analysis == analysis) & (support.hop == hop)]
            grouped = part.groupby(['center_marker', 'source_marker_name'])[['n_supported', 'n_cases']].sum()
            fraction = (grouped.n_supported / grouped.n_cases).unstack().reindex(index=names, columns=names)
            image = ax.imshow(fraction.to_numpy(), vmin=0, vmax=1, cmap='viridis')
            ax.set_xticks(range(len(names)), names, rotation=45, ha='right')
            ax.set_yticks(range(len(names)), names)
            ax.set_xlabel('Original source identity'); ax.set_ylabel('Unchanged center identity')
            ax.set_title(f'{analysis}, hop {hop}: fraction retained in paired comparison')
            for i in range(len(names)):
                for j in range(len(names)):
                    value = fraction.iloc[i, j]
                    ax.text(j, i, '—' if not np.isfinite(value) else f'{value:.0%}', ha='center', va='center',
                            color='white' if np.isfinite(value) and value < .5 else 'black', fontsize=9)
    fig.suptitle('Coverage: sweeps require support at every N; blank ≠ zero response')
    fig.subplots_adjust(wspace=.4, hspace=.45, right=.86, top=.93, bottom=.10)
    colorbar_ax = fig.add_axes([.90, .28, .015, .45])
    fig.colorbar(image, cax=colorbar_ax, label='Supported fraction')
    return fig

def replacement_plot_reference_weights(result, *, center, source, hop=1):
    """Optional inspection: identities contributing to each mixture at each N."""
    frame = result['paired_cases']
    frame = frame[(frame.analysis == 'sweep') & (frame.center_marker == center) &
                  (frame.source_marker_name == source) & (frame.hop == hop)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, mode in zip(axes, ['fixed', 'dependent']):
        columns = [f'p_{mode}_{j}' for j in range(len(result['config']['markers']))]
        if not frame.empty:
            # Average repeated seeds and cases within organoid first.
            p = frame.groupby(['evaluated_n', 'organoid_str'])[columns].mean().groupby('evaluated_n').mean()
            ax.stackplot(p.index, p.to_numpy().T, labels=result['config']['markers'])
        ax.set_xscale('log'); ax.set_ylim(0, 1); ax.set_xlabel('Supplied N'); ax.set_title(mode)
    axes[0].set_ylabel('Replacement probability, equal organoid weight')
    if not frame.empty:
        axes[-1].legend(loc='upper left', bbox_to_anchor=(1, 1))
    else:
        for ax in axes:
            ax.text(.5, .5, 'No supported cases', ha='center', transform=ax.transAxes)
    fig.suptitle(f'Center {center}, source {source}, exact hop {hop}')
    fig.tight_layout()
    return fig

## Niche study summaries

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
from pathlib import Path
import itertools
import json
import numpy as np
import pandas as pd
from src.analysis.conditioning.response_statistics import cluster_summary

NICHE_STATISTICS_METRICS = ['delta_z','delta_raw','delta_relative']  # Quantities included in the study summaries.

def niche_statistics_summarize(out, draws=1000):
    out=Path(out)
    sm=pd.read_csv(out/'recipient_manifest.csv.gz')
    pm=pd.read_csv(out/'pair_manifest.csv.gz')
    cells=pd.read_csv(out/'cell_census.csv.gz')
    config=json.loads((out/'settings.json').read_text()); counts=config['counts']
    # Sampling and descriptive prevalence refer to actual observed sizes only.
    orgs=cells[['organoid_str','observed_n','timepoint','dataset']].drop_duplicates()
    edges=np.unique(np.quantile(orgs.observed_n,[0,.25,.5,.75,1])); edges[0]-=1; edges[-1]+=1
    orgs['n_bin']=pd.cut(orgs.observed_n,edges,labels=False)
    nb=orgs.set_index('organoid_str').n_bin
    cells['n_bin']=cells.organoid_str.map(nb)
    sm['n_bin']=sm.organoid_str.map(nb)
    sm['recipient_class']=np.where(sm.recipient_lgr5==1,'LGR5+','LGR5−')
    sm['degree_bin']=(sm.recipient_degree//3).astype(int)
    sm['lgr5_bin']=np.minimum((sm.local_lgr5_fraction*4).astype(int),3)
    # Exact coarsened context strata. Same organoid controls N, timepoint and batch.
    context_keys=['organoid_str','source','hop','source_state','recipient_markers','recipient_region','degree_bin','lgr5_bin']
    context_meta=[]
    for marker, backup in [('Lysozyme','sero'),('Serotonin','lyso')]:
        m=sm[(sm.marker==marker)&(sm.recipient_lgr5==1)&(sm[f'source_{backup}']==0)].copy()
        m['backup_present']=(m[f'backup_{backup}']>0).astype(int)
        eligible=m.groupby(context_keys).backup_present.nunique()
        eligible=eligible[eligible==2].reset_index()[context_keys]
        m=m.merge(eligible,on=context_keys,validate='many_to_one')
        context_meta.append(m[['case_id','backup_present']])
    cm=pd.concat(context_meta,ignore_index=True)
    cm.to_csv(out/'backup_matched_cases.csv',index=False)
    metadata=dict(n_bin_edges=edges.tolist(),bootstrap_draws=draws,
        bootstrap_unit='organoid; fitted seeds averaged, not independent replicates',
        backup_matching=context_keys,backup_excludes_source_coexpressing_backup=True)
    (out/'summary_settings.json').write_text(json.dumps(metadata,indent=2))
    orgs.to_csv(out/'organoid_metadata.csv',index=False)
    # Cell fractions, measured curvature and marker-state availability: equal organoid weighting.
    census=[]
    for family,col in [('Lysozyme','lyso_state'),('Serotonin','sero_state')]:
        for state in ['precursor_only','double_positive','mature_only']:
            temp=cells.assign(fraction=(cells[col]==state).astype(float))
            g=temp.groupby(['organoid_str','timepoint','n_bin','region'],observed=True).agg(
                fraction=('fraction','mean'),n_cells=('fraction','size')).reset_index()
            census.append(g.assign(family=family,source_state=state))
    census=pd.concat(census,ignore_index=True)
    census.to_csv(out/'census_by_organoid.csv',index=False)
    cluster_summary(census,['timepoint','region','family','source_state'],['fraction'],draws=draws).to_csv(out/'prevalence_summary.csv',index=False)
    measured=[]
    for marker in ['LGR5','Agr2','Lysozyme','Chroma','Serotonin']:
        g=cells[cells[marker]==1].groupby(['organoid_str','timepoint','n_bin','region']).curvature.mean().reset_index()
        measured.append(g.assign(marker=marker))
    measured=pd.concat(measured,ignore_index=True)
    measured.to_csv(out/'measured_curvature_by_organoid.csv',index=False)
    cluster_summary(measured,['timepoint','region','marker'],['curvature'],draws=draws).to_csv(out/'measured_curvature_summary.csv',index=False)
    results={k:[] for k in ['field','specificity','lineage','dominance','backup','pairs','crypt','crypt_paired','timepoint','observed','recipient_types']}
    dominance_cases=[]
    support=[]
    def store(name, frame, keys, metrics):
        if len(frame):
            grouped=frame.groupby(['organoid_str',*keys],observed=True,dropna=False)[metrics].mean().reset_index()
            results[name].append(grouped)
    for fold in sorted(sm.fold.unique()):
        ms=sm[sm.fold==fold]; mp=pm[pm.fold==fold]
        for seed in config['seeds']:
            for n in counts+['observed']:
                suffix=f'fold_{fold}_seed_{seed}_N{n}'
                s=pd.read_csv(out/f'{suffix}_singles.csv.gz').merge(ms,on='case_id',validate='one_to_one')
                p=pd.read_csv(out/f'{suffix}_pairs.csv.gz').merge(mp,on='case_id',validate='one_to_one')
                assert len(s)==len(ms) and len(p)==len(mp)
                s['seed']=seed; s['n']=0 if n=='observed' else n
                p['seed']=seed; p['n']=s.n.iloc[0]
                basekeys=['seed','n']
                # Average recipients within each edited source first, then source cells within organoid.
                sourcekeys=['organoid_str','source','marker','source_state','hop','recipient_class',*basekeys]
                sf=s.groupby(sourcekeys,observed=True)[NICHE_STATISTICS_METRICS].mean().reset_index()
                store('field',sf,['marker','source_state','hop','recipient_class',*basekeys],NICHE_STATISTICS_METRICS)
                store('field',sf.assign(source_state='all_positive'),['marker','source_state','hop','recipient_class',*basekeys],NICHE_STATISTICS_METRICS)
                if n=='observed':
                    store('observed',s[s.recipient_lgr5==1],['marker','source_state','hop','n_bin',*basekeys],NICHE_STATISTICS_METRICS)
                    store('timepoint',s[s.recipient_lgr5==1],['marker','source_state','hop','timepoint',*basekeys],NICHE_STATISTICS_METRICS)
                    store('observed',s[s.recipient_lgr5==1].assign(source_state='all_positive'),['marker','source_state','hop','n_bin',*basekeys],NICHE_STATISTICS_METRICS)
                    store('timepoint',s[s.recipient_lgr5==1].assign(source_state='all_positive'),['marker','source_state','hop','timepoint',*basekeys],NICHE_STATISTICS_METRICS)
                # Paired LGR5+ minus LGR5-negative recipients around exactly the same source/hop.
                ix=['organoid_str','source','marker','source_state','hop',*basekeys]
                wide=sf.pivot(index=ix,columns='recipient_class',values=NICHE_STATISTICS_METRICS).dropna()
                contrast=pd.DataFrame({m:wide[(m,'LGR5+')]-wide[(m,'LGR5−')] for m in NICHE_STATISTICS_METRICS}).reset_index()
                for m in NICHE_STATISTICS_METRICS:
                    contrast['lgr5_'+m]=wide[(m,'LGR5+')].to_numpy()
                    contrast['other_'+m]=wide[(m,'LGR5−')].to_numpy()
                specificity_metrics=NICHE_STATISTICS_METRICS+['lgr5_'+m for m in NICHE_STATISTICS_METRICS]+['other_'+m for m in NICHE_STATISTICS_METRICS]
                store('specificity',contrast,['marker','source_state','hop',*basekeys],specificity_metrics)
                store('specificity',contrast.assign(source_state='all_positive'),['marker','source_state','hop',*basekeys],specificity_metrics)
                # Secondary recipient phenotypes overlap; LGR5+ recipients are excluded from all comparators.
                for phenotype in ['AldoB','KI67','Agr2','Chroma','Lysozyme','Serotonin']:
                    mask=(s.recipient_lgr5==0)&s.recipient_markers.fillna('').str.split('|').apply(lambda a: phenotype in a)
                    t=s[mask].groupby(['organoid_str','source','marker','hop',*basekeys])[NICHE_STATISTICS_METRICS].mean().reset_index()
                    store('recipient_types',t.assign(phenotype=phenotype),['marker','hop','phenotype',*basekeys],NICHE_STATISTICS_METRICS)
                l=s[s.recipient_lgr5==1].copy()
                store('lineage',l,['marker','source_state','hop',*basekeys],NICHE_STATISTICS_METRICS)
                store('crypt',l,['marker','source_state','hop','recipient_region',*basekeys],NICHE_STATISTICS_METRICS)
                store('lineage',l.assign(source_state='all_positive'),['marker','source_state','hop',*basekeys],NICHE_STATISTICS_METRICS)
                store('crypt',l.assign(source_state='all_positive'),['marker','source_state','hop','recipient_region',*basekeys],NICHE_STATISTICS_METRICS)
                cr=l.groupby(['organoid_str','source','marker','hop',*basekeys,'recipient_region'])[NICHE_STATISTICS_METRICS].mean().unstack('recipient_region')
                if all(region in cr.columns.levels[1] for region in ['detected_crypt','outside_detected_crypt']):
                    cr=cr.loc[:,pd.IndexSlice[:,['detected_crypt','outside_detected_crypt']]].dropna()
                    cc=pd.DataFrame({m:cr[(m,'detected_crypt')]-cr[(m,'outside_detected_crypt')] for m in NICHE_STATISTICS_METRICS}).reset_index()
                    store('crypt_paired',cc,['marker','hop',*basekeys],NICHE_STATISTICS_METRICS)
                # Mean absolute single-source sensitivity at a fixed recipient; excludes double-positive precursor edits.
                l=l[((l.marker=='Agr2')&(l.source_state=='precursor_only'))|
                    ((l.marker=='Chroma')&(l.source_state=='precursor_only'))|l.marker.isin(['Lysozyme','Serotonin'])].copy()
                for metric in NICHE_STATISTICS_METRICS: l['abs_'+metric]=l[metric].abs()
                idkeys=['organoid_str','recipient','hop',*basekeys]
                mm=l.groupby([*idkeys,'marker'])[NICHE_STATISTICS_METRICS+['abs_'+m for m in NICHE_STATISTICS_METRICS]].mean().unstack('marker')
                for a,b in itertools.combinations(['Agr2','Lysozyme','Chroma','Serotonin'],2):
                    if a not in mm.columns.levels[1] or b not in mm.columns.levels[1]: continue
                    subset=mm.loc[:,pd.IndexSlice[:,[a,b]]].dropna()
                    if not len(subset): continue
                    record=pd.DataFrame(index=subset.index)
                    for metric in NICHE_STATISTICS_METRICS:
                        av,bv=subset[('abs_'+metric,a)],subset[('abs_'+metric,b)]
                        record['a_'+metric]=subset[(metric,a)]; record['b_'+metric]=subset[(metric,b)]
                        record['strength_a_'+metric]=av; record['strength_b_'+metric]=bv
                        record['dominance_'+metric]=(av-bv)/(av+bv).where(av+bv>1e-10)
                    record=record.reset_index().assign(marker_a=a,marker_b=b)
                    columns=[c for c in record if c.endswith(tuple(NICHE_STATISTICS_METRICS))]
                    store('dominance',record,['marker_a','marker_b','hop',*basekeys],columns)
                    if n in [counts[0],counts[-1]]: dominance_cases.append(record)
                    if seed==config['seeds'][0] and n==counts[0]:
                        support.append(dict(analysis='dominance',marker_a=a,marker_b=b,hop=None,fold=fold,
                            n_recipients=len(record[['organoid_str','recipient','hop']].drop_duplicates()),n_organoids=record.organoid_str.nunique()))
                # Backup presence contrasts on matched within-organoid context strata.
                matched=s.merge(cm,on='case_id',validate='one_to_one')
                mk=[*context_keys,'marker',*basekeys]
                bm=matched.groupby([*mk,'backup_present'])[NICHE_STATISTICS_METRICS].mean().unstack('backup_present').dropna()
                if len(bm):
                    bc=pd.DataFrame({m:bm[(m,1)]-bm[(m,0)] for m in NICHE_STATISTICS_METRICS}).reset_index()
                    for m in NICHE_STATISTICS_METRICS:
                        bc['backup0_'+m]=bm[(m,0)].to_numpy(); bc['backup1_'+m]=bm[(m,1)].to_numpy()
                    store('backup',bc,['marker','hop',*basekeys],NICHE_STATISTICS_METRICS+['backup0_'+m for m in NICHE_STATISTICS_METRICS]+['backup1_'+m for m in NICHE_STATISTICS_METRICS])
                # Average source-pair draws inside recipient before organoid means.
                pmetrics=[c for c in p if c.endswith(('_z','_raw','_relative'))]
                for selection,mask in [('all',np.ones(len(p),dtype=bool)),
                    ('one_each_in_hop',(p.count_lyso==1)&(p.count_sero==1)&(p.cross_positive==0))]:
                    pp=p[mask].groupby(['organoid_str','recipient','hop',*basekeys])[pmetrics].mean().reset_index()
                    store('pairs',pp.assign(selection=selection),['selection','hop',*basekeys],pmetrics)
            print(f'Summarized fold={fold} seed={seed}',flush=True)
    pd.DataFrame(support).to_csv(out/'matched_support.csv',index=False)
    for name,parts in results.items():
        if not parts: continue
        frame=pd.concat(parts,ignore_index=True)
        frame.to_csv(out/f'{name}_organoid.csv.gz',index=False)
        keys=[c for c in frame if c not in ['organoid_str','seed'] and not c.endswith(('_z','_raw','_relative'))]
        metrics=[c for c in frame if c.endswith(('_z','_raw','_relative'))]
        cluster_summary(frame,keys,metrics,draws=draws).to_csv(out/f'{name}_summary.csv',index=False)
        frame.groupby([*keys,'seed'])[metrics].mean().reset_index().to_csv(out/f'{name}_seed_summary.csv',index=False)
        if 'n' in keys and frame.n.max()>0:
            ek=[k for k in keys if k!='n']
            # Endpoints are paired within organoid AND fitted seed before averaging seeds.
            low=frame[frame.n==counts[0]].set_index([*ek,'organoid_str','seed'])[metrics]
            high=frame[frame.n==counts[-1]].set_index([*ek,'organoid_str','seed'])[metrics]
            endpoint=(high-low).dropna(how='all').reset_index()
            cluster_summary(endpoint,ek,metrics,draws=draws).to_csv(out/f'{name}_endpoint.csv',index=False)
            endpoint.groupby([*ek,'seed'])[metrics].mean().reset_index().to_csv(out/f'{name}_endpoint_seeds.csv',index=False)
    dc=pd.concat(dominance_cases,ignore_index=True)
    dc.to_csv(out/'dominance_endpoint_cases.csv.gz',index=False)
    # A switch is an absolute-sensitivity rank change, not an additive share of the prediction.
    ix=['organoid_str','recipient','hop','marker_a','marker_b','seed']
    wide=dc.pivot(index=ix,columns='n',values='dominance_delta_z').dropna()
    switches=wide.reset_index()[ix].copy()
    switches['a_to_b']=((wide[counts[0]]>0)&(wide[counts[-1]]<0)).to_numpy().astype(float)
    switches['b_to_a']=((wide[counts[0]]<0)&(wide[counts[-1]]>0)).to_numpy().astype(float)
    switches.to_csv(out/'dominance_switch_cases.csv.gz',index=False)
    cluster_summary(switches,['hop','marker_a','marker_b'],['a_to_b','b_to_a'],draws=draws).to_csv(out/'dominance_switch_summary.csv',index=False)
    # Matching balance and support at the static case level; no model outcomes used for matching.
    balance=sm.merge(cm,on='case_id')
    balance.groupby(['marker','hop','backup_present']).agg(n_cases=('case_id','size'),n_organoids=('organoid_str','nunique'),
        mean_n=('observed_n','mean'),mean_degree=('recipient_degree','mean'),mean_lgr5_fraction=('local_lgr5_fraction','mean')).to_csv(out/'backup_balance.csv')
    covariates=['observed_n','recipient_degree','local_lgr5_fraction']
    weighted=balance.groupby([*context_keys,'marker','backup_present'])[covariates].mean().reset_index()
    weighted=weighted.groupby(['organoid_str','marker','hop','backup_present'])[covariates].mean().reset_index()
    weighted.groupby(['marker','hop','backup_present'])[covariates].mean().to_csv(out/'backup_balance_analysis_weighted.csv')
    dominance=pd.concat(results['dominance'],ignore_index=True)
    dm=['dominance_delta_z','dominance_delta_relative']
    observed=dominance[dominance.n==0].merge(orgs,on='organoid_str',validate='many_to_one')
    for stratum in ['timepoint','n_bin']:
        cluster_summary(observed,['marker_a','marker_b','hop',stratum],dm,draws=draws).to_csv(out/f'dominance_observed_{stratum}.csv',index=False)
    # Local sensitivity: nearest bracketing grid points to each graph's actual N.
    # This avoids making the full endpoint extrapolation the sole size test.
    local=[]
    actual=orgs.set_index('organoid_str').observed_n
    for keys,g in dominance[dominance.n>0].groupby(['organoid_str','marker_a','marker_b','hop','seed']):
        observed_n=actual.loc[keys[0]]
        if not counts[0] <= observed_n <= counts[-1]: continue
        g=g.sort_values('n').reset_index(drop=True)
        hi=min(max(int(np.searchsorted(g.n.to_numpy(),observed_n)),1),len(g)-1); lo=hi-1
        diff=(g.loc[hi,dm]-g.loc[lo,dm])/(np.log(g.loc[hi,'n'])-np.log(g.loc[lo,'n']))
        local.append(dict(zip(['organoid_str','marker_a','marker_b','hop','seed'],keys))|diff.to_dict())
    local=pd.DataFrame(local)
    local.to_csv(out/'dominance_local_slopes.csv',index=False)
    cluster_summary(local,['marker_a','marker_b','hop'],dm,draws=draws).to_csv(out/'dominance_local_slope_summary.csv',index=False)
    return out

## Encoding figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

ENCODING_COLORS = {
    'full_native': '#aaaaaa',  # Display color for full_native.
    'full_matched': '#2865ad',  # Display color for full_matched.
    'exclusive': '#d05a32',  # Display color for exclusive.
}  # Display colors assigned to groups or model variants.
ENCODING_LABELS = {
    'full_native': 'Full markers: original cohort',  # Figure label for full_native.
    'full_matched': 'Full markers: matched cells',  # Figure label for full_matched.
    'exclusive': 'Exclusive markers: matched cells',  # Figure label for exclusive.
}  # Display labels for figure groups or model variants.
ENCODING_MARKERS = ['Agr2','AldoB','Chroma','KI67','LGR5','Lysozyme','Serotonin']  # Marker order recorded by the selected training run.

def encoding_line(ax,g,label,color,style='-'):
    g=g[g.n>0].sort_values('n')
    if g.empty:return
    ax.plot(g.n,g['mean'],style,color=color,lw=1.8,label=label)
    ax.fill_between(g.n,g.ci_low,g.ci_high,color=color,alpha=.11)

def encoding_axis(ax,ylabel='Normalized curvature change',xlabel='Supplied N'):
    ax.set_xscale('log');ax.set_xticks([165,244,361,535,791],[165,244,361,535,791]);ax.xaxis.set_minor_formatter(NullFormatter())
    ax.axhline(0,color='.5',lw=.7);ax.set_xlabel(xlabel);ax.set_ylabel(ylabel);ax.grid(alpha=.15)

def encoding_figures(comparison):
    comparison=Path(comparison);table=comparison/'comparison_tables';items={}
    def read(name):return pd.read_csv(table/f'{name}.csv')
    audit=pd.read_csv(comparison/'exclusive/tables/exclusivity_audit.csv').groupby('marker')[['full_positive','exclusive_positive']].sum().reindex(ENCODING_MARKERS)
    fig,ax=plt.subplots(figsize=(10,4),layout='constrained');xx=np.arange(7)
    ax.bar(xx-.2,audit.full_positive,width=.4,label='Full markers',color=ENCODING_COLORS['full_matched'])
    ax.bar(xx+.2,audit.exclusive_positive,width=.4,label='Exclusive markers',color=ENCODING_COLORS['exclusive'])
    ax.set_xticks(xx,ENCODING_MARKERS);ax.set_ylabel('Positive cells');ax.set_title('Same cells and seven feature columns; ordered suppression');ax.legend()
    items['01_encoding_audit']=fig
    overall=read('mse_overall');binned=read('mse_by_size')
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for i,encoding in enumerate(['full','exclusive']):
        c=ENCODING_COLORS['full_matched' if encoding=='full' else encoding];g=overall[overall.encoding==encoding].iloc[0]
        axes[0].bar(i,g['mean'],color=c);axes[0].vlines(i,g.ci_low,g.ci_high,color='black')
        q=binned[binned.encoding==encoding];encoding_line(axes[1],q,encoding,c)
    axes[0].set_xticks([0,1],['Full','Exclusive']);axes[0].set_ylabel('Validation MSE (physical curvature²)')
    axes[0].set_title('Equal organoid weighting; same targets and folds')
    encoding_axis(axes[1],'Validation MSE (physical curvature²)','Observed N (bin median)');axes[1].legend()
    items['02_validation_mse']=fig
    d=read('ablation_summary');d=d[d.metric=='delta_relative']
    for mode in ['sweep','observed']:
        for hop in [1,2]:
            fig,axes=plt.subplots(8,7,figsize=(20,20),layout='constrained',sharex=True)
            for row,center in enumerate([*ENCODING_MARKERS,'unmarked']):
                for col,source in enumerate(ENCODING_MARKERS):
                    ax=axes[row,col];q=d[(d['mode']==mode)&(d.hop==hop)&(d.center_marker==center)&(d.source_marker_name==source)]
                    for encoding in ['full_native','full_matched','exclusive']:
                        g=q[(q.encoding==encoding)&(q.n_organoids>=5)]
                        encoding_line(ax,g,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
                    encoding_axis(ax,center if col==0 else '',('Observed N' if mode=='observed' else 'Supplied N') if row==7 else '')
                    ax.tick_params(labelsize=7)
                    if row==0:ax.set_title(source,fontsize=10)
            handles,labels=axes[0,0].get_legend_handles_labels()
            fig.legend(handles,labels,loc='outside lower center',ncol=3,fontsize=10)
            fig.suptitle(f'{mode.title()} size, hop {hop}: rows = unchanged recipient marker; columns = edited source marker\nNormalized effects; panels mask fewer than 5 organoids',fontsize=14)
            items[f'03_{mode}_all_markers_hop{hop}']=fig
    fig,axes=plt.subplots(2,4,figsize=(16,8),layout='constrained')
    for row,hop in enumerate([1,2]):
        for col,source in enumerate(['Agr2','Lysozyme','Chroma','Serotonin']):
            ax=axes[row,col];q=d[(d['mode']=='sweep')&(d.hop==hop)&(d.center_marker=='LGR5')&(d.source_marker_name==source)]
            for encoding in ['full_native','full_matched','exclusive']:
                encoding_line(ax,q[(q.encoding==encoding)&(q.n_organoids>=5)],ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
            encoding_axis(ax);ax.set_title(f'{source} → LGR5, hop {hop}')
    handles,labels=axes[0,0].get_legend_handles_labels();fig.legend(handles,labels,loc='outside lower center',ncol=3)
    items['04_lgr5_sweep_comparison']=fig
    fig,axes=plt.subplots(2,4,figsize=(16,8),layout='constrained')
    z=read('ablation_summary')
    for row,encoding in enumerate(['full_matched','exclusive']):
        for col,source in enumerate(['Agr2','Lysozyme','Chroma','Serotonin']):
            ax=axes[row,col];q=z[(z.encoding==encoding)&(z.hop==1)&(z.center_marker=='LGR5')&(z.source_marker_name==source)&(z.metric=='delta_relative')]
            for mode,color in [('observed','#2865ad'),('sweep','#d05a32')]:encoding_line(ax,q[q['mode']==mode],mode,color)
            encoding_axis(ax,xlabel='Observed-bin or supplied N');ax.set_title(f'{encoding}: {source} → LGR5');ax.legend(fontsize=8)
    items['05_observed_and_sweep']=fig
    fig,axes=plt.subplots(1,4,figsize=(16,4),layout='constrained')
    for ax,source in zip(axes,['Agr2','Lysozyme','Chroma','Serotonin']):
        q=d[(d['mode']=='sweep')&(d.hop==1)&(d.center_marker=='LGR5')&(d.source_marker_name==source)]
        for encoding,label,color in [('full_matched','Full model: remove one marker','#2865ad'),
            ('full_fate_control','Full model: remove all source fate','#39925b'),('exclusive','Exclusive model: remove sole marker','#d05a32')]:
            encoding_line(ax,q[q.encoding==encoding],label,color)
        encoding_axis(ax);ax.set_title(f'{source} source → LGR5 recipient')
    fig.legend(*axes[0].get_legend_handles_labels(),loc='outside lower center',ncol=3)
    items['05b_full_source_fate_control']=fig
    # Matched niche panels are generated once all checkpoint diagnostics complete.
    if not (table/'niche_dominance_summary.csv').exists():return items
    d=read('niche_dominance_summary')
    pairs=[('Agr2','Lysozyme'),('Chroma','Serotonin'),('Lysozyme','Serotonin')]
    fig,axes=plt.subplots(2,3,figsize=(15,8),layout='constrained')
    for row,hop in enumerate([1,2]):
        for col,(a,b) in enumerate(pairs):
            ax=axes[row,col];q=d[(d.marker_a==a)&(d.marker_b==b)&(d.hop==hop)&(d.metric=='dominance_delta_relative')]
            for encoding in ENCODING_COLORS:encoding_line(ax,q[q.encoding==encoding],ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
            encoding_axis(ax,f'D: positive favors {a}');ax.set_ylim(-1,1);ax.set_title(f'{a} vs {b}, hop {hop}')
    fig.legend(*axes[0,0].get_legend_handles_labels(),loc='outside lower center',ncol=3)
    fig.suptitle('Relative single-source sensitivity at LGR5 recipients; common positive normalization cancels in D')
    items['06_niche_dominance']=fig
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout='constrained')
    for ax,(a,b) in zip(axes,pairs):
        for encoding in ['full_matched','exclusive']:
            for metric,style in [('dominance_delta_relative','-'),('dominance_delta_z','--')]:
                q=d[(d.marker_a==a)&(d.marker_b==b)&(d.hop==1)&(d.metric==metric)&(d.encoding==encoding)]
                encoding_line(ax,q,encoding+(' normalized' if style=='-' else ' transformed'),ENCODING_COLORS[encoding],style)
        encoding_axis(ax,f'D: positive favors {a}');ax.set_title(f'{a} vs {b}');ax.legend(fontsize=7)
    items['07_dominance_output_scale_check']=fig
    obs=read('niche_dominance_observed_n_bin');local=read('niche_dominance_local_slope_summary')
    reference=Path(json.loads((comparison/'comparison_settings.json').read_text())['reference_run'])
    orgs=pd.read_csv(reference/'niche_hypotheses/organoid_metadata.csv')
    obscenters=orgs.groupby('n_bin').observed_n.median().to_dict()
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,(a,b) in zip(axes,pairs[:2]):
        for encoding in ENCODING_COLORS:
            q=obs[(obs.marker_a==a)&(obs.marker_b==b)&(obs.hop==1)&(obs.metric=='dominance_delta_relative')&(obs.encoding==encoding)].copy()
            q['n']=q.n_bin.map(obscenters);encoding_line(ax,q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
        encoding_axis(ax,f'D: positive favors {a}','Observed N (quartile median)');ax.set_title(f'{a} vs {b}');ax.legend(fontsize=7)
    items['08_observed_niche_dominance']=fig
    d=read('niche_pairs_summary');backup=read('niche_backup_summary')
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for col,hop in enumerate([1,2]):
        for encoding in ENCODING_COLORS:
            q=d[(d.hop==hop)&(d.selection=='all')&(d.metric=='interaction_relative')&(d.encoding==encoding)]
            encoding_line(axes[0,col],q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
            q=d[(d.hop==hop)&(d.selection=='all')&(d.metric=='interaction_z')&(d.encoding==encoding)]
            encoding_line(axes[1,col],q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
        control=pd.read_csv(comparison/'full_fate_pairs/pairs_summary.csv')
        for row,metric in [(0,'interaction_relative'),(1,'interaction_z')]:
            q=control[(control.hop==hop)&(control.selection=='all')&(control.metric==metric)]
            encoding_line(axes[row,col],q,'Full model: remove whole source vectors','#39925b')
        encoding_axis(axes[0,col],'Factorial interaction, normalized');encoding_axis(axes[1,col],'Factorial interaction, transformed')
        axes[0,col].set_title(f'Lysozyme × Serotonin around unchanged LGR5, hop {hop}')
    fig.legend(*axes[0,0].get_legend_handles_labels(),loc='outside lower center',ncol=2)
    items['09_redundancy_comparison']=fig
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,marker in zip(axes,['Lysozyme','Serotonin']):
        for encoding in ENCODING_COLORS:
            q=backup[(backup.marker==marker)&(backup.hop==1)&(backup.metric=='delta_z')&(backup.encoding==encoding)]
            encoding_line(ax,q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
        encoding_axis(ax,'Backup present − absent effect, transformed');ax.set_title(f'Edit {marker}: same-source context matching');ax.legend(fontsize=7)
    items['10_backup_context']=fig
    for name,metric,ylabel,title in [('specificity','delta_relative','LGR5+ minus LGR5-negative effect','Matched recipient specificity'),
        ('crypt_paired','delta_relative','Inside minus outside detected crypt effect','Same-source detected-crypt context')]:
        d=read('niche_'+name+'_summary');fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        if name=='specificity':d=d[d.source_state=='all_positive']
        for ax,marker in zip(axes,['Lysozyme','Serotonin']):
            for encoding in ENCODING_COLORS:
                q=d[(d.marker==marker)&(d.hop==1)&(d.metric==metric)&(d.encoding==encoding)]
                encoding_line(ax,q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding],':' if encoding=='full_native' else '-')
            encoding_axis(ax,ylabel);ax.set_title(f'{title}: {marker}, hop 1');ax.legend(fontsize=7)
        items['11_'+name]=fig
    d=read('lgr5_effect_summary')
    fig,axes=plt.subplots(2,4,figsize=(16,8),layout='constrained')
    for row,encoding in enumerate(['full_matched','exclusive']):
        for col,marker in enumerate(['Agr2','Lysozyme','Chroma','Serotonin']):
            ax=axes[row,col]
            for route,color in zip(['full','film_only','head_only','layer1_only','layer2_only'],plt.cm.tab10.colors):
                q=d[(d.encoding==encoding)&(d.source_marker_name==marker)&(d.hop==1)&(d.metric=='delta_z')&(d.route==route)]
                encoding_line(ax,q,route,color)
            encoding_axis(ax,'Transformed curvature change');ax.set_title(f'{encoding}: {marker} → LGR5');ax.legend(fontsize=7)
    items['12_conditioning_routes']=fig
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,hop in zip(axes,[1,2]):
        for encoding,style in [('full_matched','-'),('exclusive','--')]:
            for marker,color in zip(['Lysozyme','Serotonin'],['#2865ad','#d05a32']):
                q=d[(d.encoding==encoding)&(d.source_marker_name==marker)&(d.hop==hop)&(d.metric=='h2_cosine')&(d.route=='full')]
                encoding_line(ax,q,f'{encoding}: {marker}',color,style)
        encoding_axis(ax,'Cosine to own N=361 hidden-response direction');ax.set_title(f'Within-model representation change, hop {hop}');ax.legend(fontsize=7)
    items['13_hidden_direction']=fig
    common=read('lgr5_common_scaling')
    fig,axes=plt.subplots(1,3,figsize=(15,4),layout='constrained')
    for ax,(a,b) in zip(axes,[('LGR5','Lysozyme'),('Chroma','Serotonin'),('Lysozyme','Serotonin')]):
        for encoding in ['full_matched','exclusive']:
            q=common[(common.encoding==encoding)&(common.route=='full')&(common.hop==1)&
                (common.metric=='delta_z')&(common.marker_a==a)&(common.marker_b==b)].rename(columns={'mismatch':'mean','mismatch_low':'ci_low','mismatch_high':'ci_high'})
            encoding_line(ax,q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding])
        encoding_axis(ax,'Response norm not explained by common gain');ax.set_title(f'{a} / {b} profile, hop 1');ax.legend(fontsize=7)
    items['14_common_scaling_check']=fig
    interaction=read('lgr5_interaction_summary')
    fig,axes=plt.subplots(1,4,figsize=(16,4),layout='constrained')
    for ax,marker in zip(axes,['Agr2','Lysozyme','Chroma','Serotonin']):
        for encoding in ['full_matched','exclusive']:
            q=interaction[(interaction.encoding==encoding)&(interaction.kind=='center_neighbor')&
                (interaction.marker_b==marker)&(interaction.hop==1)&(interaction.metric=='interaction_z')]
            encoding_line(ax,q,ENCODING_LABELS[encoding],ENCODING_COLORS[encoding])
        encoding_axis(ax,'Factorial interaction, transformed');ax.set_title(f'Center LGR5 × neighbor {marker}');ax.legend(fontsize=7)
    fig.suptitle('Secondary diagnostic: center identity is removed here; perturbation severity differs across encodings')
    items['15_center_removal_secondary']=fig
    return items

def encoding_save_figures(comparison):
    comparison=Path(comparison);folder=comparison/'figures';folder.mkdir(exist_ok=True)
    items=encoding_figures(comparison)
    for name,fig in items.items():
        fig.savefig(folder/f'{name}.png',dpi=150,bbox_inches='tight');fig.savefig(folder/f'{name}.pdf',bbox_inches='tight')
    return items

# Exclusive-fate replacement ablation

Compare three **single-source-cell** interventions with the same unchanged recipient, graph and supplied size:

1. **Zero → unassigned:** the previous marker ablation, which is identity replacement in an exclusive vector.
2. **Context-matched replacement, fixed weights:** average discrete replacement predictions with weights fitted at the source organoid's **observed N**, held fixed during the sweep.
3. **Context-matched replacement, N-dependent weights:** recompute the reference mixture at each **supplied N**.

All seven measured fates and **Unassigned** are included as recipient and source identities. An unassigned source has a zero old-ablation effect by construction; replacing it with an assigned identity can have an effect. Static means inference at each organoid's observed size, not removal of size conditioning. At observed N, the two replacement variants coincide exactly.

This is a fresh analysis across all identity combinations, without selecting a particular marker pair, size minimum, or biological hypothesis. Existing exclusive FiLM checkpoints are reused; **no training occurs**.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Repository root used to resolve data and result paths.
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent  # Repository root used to resolve data and result paths.
assert (ROOT / 'src').is_dir(), 'Open this notebook inside the GraphNN project.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis.interventions.replacement import MatchConfig, load_comparison
# Study-specific functions are defined in this notebook.

## Settings to inspect

Matching uses full-graph source context, even when prediction is evaluated on an exact two-hop recipient ego. Donors come only from the model's training fold. A donor must have a different identity and have the recipient identity in its exact hop-1 or hop-2 ring.

Within that pool, nearest-neighbor matching uses log(1 + degree), eight neighboring identity fractions, and log N. One distance unit corresponds to a degree-plus-one ratio of two, a 25 percentage-point change in one fraction, or a size ratio of two. The candidate's own identity is excluded from the descriptor. Hard calipers and minimum donor counts reject weakly supported matches. No curvature, predictions, crypt labels, geometry, or timepoint labels are used for matching.

Each selected donor organoid has equal total weight. Low-support replacement identities are removed, and the remaining probabilities are renormalized. The original source identity is always excluded; this estimates **replacement by a different identity**, not unconditional identity imputation. Unassigned is an ordinary donor identity. These defaults are adjustable descriptive matching choices, not a calibrated model of biological conversions.

In [ ]:
# Paths and outputs
from src.artifacts.paths import resolve_training_run
from src.artifacts.size_reference import size_reference
from src.artifacts.runs import AnalysisRun
TRAINING_RUN = None  # Run name in training_results/gin_depth_training, or an explicit historical path.
REFERENCE_MODEL_KEY = None  # One FiLM catalog key selecting the depth/width grid for interventions.
TRAINED_RUN = resolve_training_run(ROOT, 'gin_depth_training', TRAINING_RUN)
display(AnalysisRun(TRAINED_RUN).records)
RUN_DIR = size_reference(TRAINED_RUN, REFERENCE_MODEL_KEY, root=ROOT)
OUTPUT_DIR = TRAINED_RUN / 'analysis' / 'replacement_ablation'  # Directory for this analysis’s tables and figures.

# None selects all saved folds / all model seeds, respectively.

# Saved model selection
FOLDS = None  # Saved folds to use; None selects all folds.
MODEL_SEEDS = None  # Model initialization seeds included in this experiment.

# Size sweeps
SWEEP_COUNTS = json.loads((RUN_DIR / 'sweep_grid.json').read_text())  # Supplied cell counts to evaluate; None uses the saved grid.

# Ablation sampling
HOPS = (1, 2)  # Exact graph-distance rings to analyze.
CENTERS_PER_IDENTITY = 2  # Recipient quota per identity in each organoid.
SWEEP_CENTERS_PER_IDENTITY = 1  # Per-identity recipient quota for N sweeps.
SAMPLING_SEED = 1701  # Random seed for reproducible sampling or initialization.

# Inference resources
BATCH_SIZE = 128  # Number of graphs or subgraphs per batch.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.
CPU_THREADS = 4  # Maximum CPU threads used for inference.

MATCHING = MatchConfig(
    # Replacement matching
    neighbors=256,  # Number of candidate donor neighbors in context matching.
    max_size_ratio=2.0,  # Largest source/donor cell-count ratio allowed in matching.
    max_degree_ratio=2.0,  # Largest source/donor degree ratio allowed in matching.
    max_composition_l1=1.0,  # Largest allowed L1 neighborhood-composition distance.

    # Donor support and weighting
    min_cells=20,  # Minimum donor cells needed for replacement support.
    min_organoids=5,  # Minimum distinct organoids required for reported effects.
    min_identity_cells=5,  # Minimum donor cells per replacement identity.
    min_identity_organoids=3,  # Minimum donor organoids per replacement identity.
    max_donors_per_organoid=256,  # Cap each organoid’s contribution to the donor pool.
)  # Context-matching and support settings for replacement donors.

# Statistical summaries
BOOTSTRAP_SAMPLES = 500  # Number of organoid bootstrap resamples.
MIN_PLOT_ORGANOIDS = 5  # Minimum organoid support for a plotted curve point.

# Display options
PLOT_METRIC = 'delta_relative'  # Effect scale: delta_relative, delta_mu, or delta_z.

# Inference is resumable; existing matching settings/checkpoints/code must agree.
# Set False to redisplay a completed analysis without running inference.

# Execution switches
RUN_INFERENCE = False  # Evaluate checkpoints to generate new intervention tables.
print('Device:', DEVICE)
print('Sweep N:', SWEEP_COUNTS)
print('Matching:', MATCHING)
print('Output:', OUTPUT_DIR)

## Inference and auditable outputs

Predictions are enumerated for every discrete replacement identity, so there is no Monte Carlo replacement noise. Each replacement prediction is converted back to curvature **before** applying its probability weight. All methods use the same sampled source cells and unchanged recipient identities.

Outputs include donor manifests and training/validation membership, source-case manifests, per-identity donor support, fixed and N-dependent weights, all individual replacement contrasts, and the three averaged contrasts. Unsupported mixtures are missing values; no global-frequency fallback is used. Changing inference settings or code requires a new output directory. Interrupted inference resumes completed fold/seed/N shards.

This cell performs potentially substantial inference over all checkpoints and sizes. It never retrains a model.

In [ ]:
from src.artifacts.provenance import workflow_fingerprint
from dataclasses import asdict, dataclass
import copy
import hashlib
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from src.analysis.interventions.perturbation import predict_subgraph_center_distribution
from src.analysis.interventions.size_sweeps import _size_override, summarize_by_organoid
from src.data.subgraphs import build_ego_subgraphs_for_graph
from src.graph.neighborhood import compute_hop_rings
from src.analysis.interventions.replacement import MatchConfig, identity_codes, describe_graph, ReplacementMatcher, make_cases, replacement_weights, evaluate_replacements, contrast_table, _write_csv, load_comparison, UNASSIGNED, METHODS
WORKFLOW_SOURCE = PROJECT_ROOT / 'experiments/ablation/fate_interventions.ipynb'  # Notebook used when recording the code fingerprint.


### Run replacement analysis

The study sequence is defined here; shared numerical operations are imported above.

In [ ]:
from src.artifacts import pickle_compat as artifact_pickle
def run_replacement_analysis(root, run_dir, output_dir, *, counts=None, folds=None, seeds=None,
        matcher_config=MatchConfig(), centers_per_identity=2, sweep_centers_per_identity=1,
        hops=(1, 2), sampling_seed=1701, device=None, batch_size=128):
    """Resumable inference only; refuses to reuse an output with different settings."""
    from src.analysis.size_conditioning.cohort_inputs import load_cohort, fold_graphs, make_model
    from src.data.metadata import load_marker_names_from_dir

    root, run_dir, out = map(lambda p: Path(p).resolve(), (root, run_dir, output_dir))
    settings = json.loads((run_dir / 'settings.json').read_text())
    if settings.get('FEATURE_ENCODING') != 'exclusive_ordered':
        raise ValueError('Use an exclusive-fate FiLM checkpoint run.')
    if settings['MODEL_GLOBAL_FEATURES']['gin_film_size'] != ['log_num_cells']:
        raise ValueError('Expected FiLM with log(N) only in the head.')
    counts = list(counts if counts is not None else json.loads((run_dir / 'sweep_grid.json').read_text()))
    if not counts or any(not np.isfinite(n) or n <= 0 for n in counts) or len(set(counts)) != len(counts):
        raise ValueError('Sweep counts must be unique, positive and finite.')
    counts = sorted(float(n) for n in counts)
    splits = json.loads((run_dir / 'splits.json').read_text())
    folds = list(folds if folds is not None else [s['fold'] for s in splits])
    seeds = list(seeds if seeds is not None else settings['MODEL_SEEDS'])
    if not folds or not seeds or not set(folds) <= {s['fold'] for s in splits} or not set(seeds) <= set(settings['MODEL_SEEDS']):
        raise ValueError('Invalid folds or seeds.')
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = root / 'training_data' / settings['DATASET_NAME']
    markers = settings.get('MARKER_NAMES') or load_marker_names_from_dir(str(dataset))
    data_signature = hashlib.sha256(json.dumps([
        (p.name, p.stat().st_size, p.stat().st_mtime_ns)
        for p in (sorted(dataset.iterdir()) if not (run_dir/'cohort/manifest.json').exists() else []) if p.suffix in {'.npz', '.json'}
    ]).encode()).hexdigest()
    artifacts = [run_dir / name for name in ('settings.json', 'splits.json', 'geometric_normalization/references.csv')]
    if (run_dir/'cohort/manifest.json').exists():
        artifacts.append(run_dir/'cohort/manifest.json')
    artifacts += [run_dir / f'checkpoints/fold_{f}_preprocessing.pkl' for f in folds]
    artifacts += [run_dir / f'checkpoints/fold_{f}_seed_{s}_gin_film_size.pt' for f in folds for s in seeds]
    config = dict(run_dir=str(run_dir), dataset=str(dataset), counts=counts, folds=folds, seeds=seeds,
        markers=markers + [UNASSIGNED], matcher=asdict(matcher_config), centers_per_identity=centers_per_identity,
        sweep_centers_per_identity=sweep_centers_per_identity, hops=list(hops), sampling_seed=sampling_seed,
        device=str(device), batch_size=batch_size, dataset_stat_sha256=data_signature,
        code_sha256=workflow_fingerprint(WORKFLOW_SOURCE),
        artifact_sha256={str(p.relative_to(run_dir)): hashlib.sha256(p.read_bytes()).hexdigest() for p in artifacts})
    out.mkdir(parents=True, exist_ok=True)
    config_path = out / 'settings.json'
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise ValueError('Output settings/code/checkpoints changed. Choose a new output directory.')
    config_path.write_text(json.dumps(config, indent=2))
    for folder in ('tables', 'support', 'figures'):
        (out / folder).mkdir(exist_ok=True)
    refs = pd.read_csv(run_dir / 'geometric_normalization/references.csv').set_index('fold')
    graphs = load_cohort(dataset, settings, run_dir)
    descriptions = [describe_graph(g) for g in graphs]
    for split in splits:
        fold = split['fold']
        if fold not in folds:
            continue
        print(f'Preparing replacement references: fold {fold}', flush=True)
        rng = np.random.default_rng(sampling_seed + fold)
        train_orgs = [graphs[i].organoid_str for i in split['train_indices']]
        val_orgs = [graphs[i].organoid_str for i in split['val_indices']]
        donor_parts = []
        for i in split['train_indices']:
            desc = descriptions[i]
            chosen = rng.choice(len(desc), min(len(desc), matcher_config.max_donors_per_organoid), replace=False)
            donor_parts.append(desc.iloc[chosen])
        donors = pd.concat(donor_parts, ignore_index=True)
        matcher = ReplacementMatcher(donors, len(markers) + 1, matcher_config, validation_organoids=val_orgs)
        _write_csv(donors, out / 'support' / f'fold_{fold}_donors.csv.gz')
        (out / 'support' / f'fold_{fold}_membership.json').write_text(json.dumps(dict(train=train_orgs, val=val_orgs)))
        with (run_dir / f'checkpoints/fold_{fold}_preprocessing.pkl').open('rb') as f:
            pre = artifact_pickle.load(f)
        groups = fold_graphs(graphs, split, pre, settings)
        subs, cases, contexts = make_cases(groups['val'], markers, hops=hops,
            centers_per_identity=centers_per_identity, sweep_centers_per_identity=sweep_centers_per_identity,
            seed=sampling_seed + fold, receptive_hops=max(2, settings['NUM_LAYERS']))
        _write_csv(cases, out / 'support' / f'fold_{fold}_cases.csv.gz')
        fixed, audit = replacement_weights(cases, contexts, matcher)
        _write_csv(audit, out / 'support' / f'fold_{fold}_observed.csv.gz')
        sweep_mask = cases.sweep.to_numpy(bool)
        sweep_cases = cases[sweep_mask].reset_index(drop=True)
        sweep_contexts = [c for c, keep in zip(contexts, sweep_mask) if keep]
        references = {None: (cases, fixed, fixed)}
        for n in counts:
            dependent, audit = replacement_weights(sweep_cases, sweep_contexts, matcher, count=n)
            _write_csv(audit, out / 'support' / f'fold_{fold}_N{n:g}.csv.gz')
            references[n] = (sweep_cases, fixed[sweep_mask], dependent)
        # Release training graphs and neighbor-search indices before GPU inference.
        del groups, matcher
        for seed in seeds:
            model = make_model(settings, markers)
            model.load_state_dict(torch.load(run_dir / f'checkpoints/fold_{fold}_seed_{seed}_gin_film_size.pt',
                                             map_location='cpu', weights_only=True))
            for n, (selected, p_fixed, p_dependent) in references.items():
                label = 'observed' if n is None else f'N{n:g}'
                path = out / 'tables' / f'fold_{fold}_seed_{seed}_{label}.csv.gz'
                if path.exists():
                    continue
                predictions = evaluate_replacements(subs, selected, model, pre['residual_transform'],
                    size_center=pre['size_center'], size_scale=pre['size_scale'], count=n,
                    batch_size=batch_size, device=device)
                frame = contrast_table(selected, predictions, p_fixed, p_dependent, count=n,
                    fold=fold, seed=seed, reference=refs.loc[fold])
                _write_csv(frame, path)
                print(f'Complete fold={fold}, seed={seed}, {label}: {len(frame)} cases', flush=True)
            del model
        del subs
    (out / 'complete.json').write_text(json.dumps(dict(folds=folds, seeds=seeds, counts=counts)))
    return out

In [ ]:
from src.artifacts.provenance import workflow_fingerprint
import json
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import torch
from src.analysis.interventions.masking import ObservedFateAdapter, evaluate_single_mask, load_mask_checkpoint, checkpoint_path, checked_settings, _sha
from src.analysis.size_conditioning.cohort_inputs import load_cohort, fold_graphs
from src.analysis.interventions.replacement import evaluate_replacements, contrast_table, _write_csv, load_comparison
from src.analysis.interventions.size_sweeps import summarize_by_organoid
from src.data.subgraphs import build_ego_subgraphs_for_graph
from src.analysis.interventions.masking_comparison import reconstruct_cases, load_masking_comparison, MASK_METHODS
WORKFLOW_SOURCE = PROJECT_ROOT / 'experiments/ablation/fate_interventions.ipynb'  # Notebook used when recording the code fingerprint.


### Run masking ablation

The study sequence is defined here; shared numerical operations are imported above.

In [ ]:
from src.artifacts import pickle_compat as artifact_pickle
def run_masking_ablation(root, replacement_dir, masking_run, output_dir, *, rate=.02,
                          batch_size=128, device=None):
    root,old,training,out=map(lambda p:Path(p).resolve(),(root,replacement_dir,masking_run,output_dir))
    if not (old/'complete.json').exists() or not (training/'training_complete.json').exists():
        raise RuntimeError('Complete replacement inference and masking training first.')
    oldcfg=json.loads((old/'settings.json').read_text()); cfg=json.loads((training/'settings.json').read_text())
    if rate<=0 or rate not in cfg['training']['rates']: raise ValueError('Select a trained positive masking rate.')
    if not set(oldcfg['folds'])<=set(cfg['folds']) or not set(oldcfg['seeds'])<=set(cfg['seeds']):
        raise ValueError('Masking models must cover every replacement fold and seed.')
    if Path(oldcfg['run_dir']).resolve()!=Path(cfg['reference_run']).resolve():
        raise ValueError('Replacement and masking runs must use the same source experiment.')
    if oldcfg['markers'][:-1]!=cfg['markers']: raise ValueError('Marker order mismatch.')
    for name in ['settings.json','splits.json','geometric_normalization/references.csv',
                 *[f'checkpoints/fold_{fold}_preprocessing.pkl' for fold in oldcfg['folds']]]:
        if oldcfg['artifact_sha256'][name] != cfg['dependencies'][name]:
            raise ValueError(f'Replacement and masking preprocessing differ: {name}')
    settings=cfg['model_settings']; markers=cfg['markers']; reference=training/'reference'
    device=device or ('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint_hashes={checkpoint_path(training,f,s,rate).name:_sha(checkpoint_path(training,f,s,rate))
                       for f in oldcfg['folds'] for s in oldcfg['seeds']}
    old_files=sorted((old/'tables').glob('*.csv.gz'))+sorted((old/'support').glob('*_cases.csv.gz'))
    checked_settings(out/'settings.json',dict(replacement_dir=str(old),masking_run=str(training),mask_rate=rate,
        folds=oldcfg['folds'],seeds=oldcfg['seeds'],counts=oldcfg['counts'],markers=oldcfg['markers'],
        code_sha256=workflow_fingerprint(WORKFLOW_SOURCE),model_code_sha256=_sha(PROJECT_ROOT / 'src/data/fate_masking.py'),
        training_settings_sha256=_sha(training/'settings.json'),reference_settings_sha256=_sha(old/'settings.json'),
        reference_files={str(p.relative_to(old)):_sha(p) for p in old_files},checkpoints=checkpoint_hashes))
    (out/'tables').mkdir(exist_ok=True);(out/'figures').mkdir(exist_ok=True)
    graphs=load_cohort(root/'training_data'/settings['DATASET_NAME'],settings,reference)
    refs=pd.read_csv(reference/'geometric_normalization/references.csv').set_index('fold')
    for split in json.loads((reference/'splits.json').read_text()):
        fold=split['fold']
        if fold not in oldcfg['folds']: continue
        with (reference/f'checkpoints/fold_{fold}_preprocessing.pkl').open('rb') as f: pre=artifact_pickle.load(f)
        val=fold_graphs(graphs,split,pre,settings)['val']
        manifest=pd.read_csv(old/'support'/f'fold_{fold}_cases.csv.gz')
        subs,cases=reconstruct_cases(val,manifest,receptive_hops=max(2, settings['NUM_LAYERS']))
        for seed in oldcfg['seeds']:
            model=load_mask_checkpoint(training,fold,seed,rate,settings,markers)
            for n in [None,*oldcfg['counts']]:
                label='observed' if n is None else f'N{n:g}'
                filename=f'fold_{fold}_seed_{seed}_{label}.csv.gz';path=out/'tables'/filename
                if path.exists(): continue
                prior=pd.read_csv(old/'tables'/filename)
                selected=cases.set_index('case_id').loc[prior.case_id].reset_index()
                for col in ['organoid_str','orig_center','orig_source_node','source_identity','center_identity','hop','observed_n']:
                    if not np.array_equal(selected[col].to_numpy(),prior[col].to_numpy()):
                        raise ValueError(f'Case identity mismatch: {col}')
                k=len(markers)+1
                fixed=prior[[f'p_fixed_{j}' for j in range(k)]].to_numpy()
                dependent=prior[[f'p_dependent_{j}' for j in range(k)]].to_numpy()
                # Known-fate replacements never set the missingness flag.
                pred=evaluate_replacements(subs,selected,ObservedFateAdapter(model),pre['residual_transform'],
                    size_center=pre['size_center'],size_scale=pre['size_scale'],count=n,batch_size=batch_size,device=device)
                frame=contrast_table(selected,pred,fixed,dependent,count=n,fold=fold,seed=seed,reference=refs.loc[fold])
                missing=evaluate_single_mask(subs,selected,model,pre['residual_transform'],size_center=pre['size_center'],
                    size_scale=pre['size_scale'],count=n,batch_size=batch_size,device=device,include_zero=False)
                np.testing.assert_allclose(frame.base_mu,missing.intact_mu,atol=1e-7,rtol=1e-5)
                frame['masking_delta_mu']=missing.mask_delta_mu
                frame['masking_delta_z']=missing.mask_delta_z
                frame['masking_delta_relative']=missing.mask_delta_mu*frame.normalization_factor
                frame['mask_rate']=rate
                if n is None:
                    frame['masking_delta_mse_z']=missing.delta_mse_z
                    frame['masking_delta_mse_mu']=missing.delta_mse_mu
                _write_csv(frame,path)
                print(f'Masking ablation fold={fold} seed={seed}, {label}',flush=True)
    (out/'complete.json').write_text('{}')
    return out

In [ ]:
torch.set_num_threads(CPU_THREADS)
if RUN_INFERENCE:
    run_replacement_analysis(
        ROOT, RUN_DIR, OUTPUT_DIR,
        counts=SWEEP_COUNTS, folds=FOLDS, seeds=MODEL_SEEDS,
        matcher_config=MATCHING,
        centers_per_identity=CENTERS_PER_IDENTITY,
        sweep_centers_per_identity=SWEEP_CENTERS_PER_IDENTITY,
        hops=HOPS, sampling_seed=SAMPLING_SEED,
        device=DEVICE, batch_size=BATCH_SIZE,
    )
result = load_comparison(OUTPUT_DIR, bootstrap_samples=BOOTSTRAP_SAMPLES)
print('Unique sampled cases:', result['cases'][['fold', 'case_id']].drop_duplicates().shape[0])
print('Identities:', result['config']['markers'])

## Support comes first

For a fair three-method comparison, static plots use cases with replacement support at observed N. Sweep plots use a **constant cohort** supported by both reference schemes at **every** sweep N. The old zero curve is recomputed on exactly that same cohort. This prevents a changing sample from creating a size trend. It also means these curves need not equal earlier notebook averages, which used different sampling and support.

The coverage figure reports the fraction of sampled cases retained. Rare or unsupported combinations remain blank rather than being reported as zero. Inspect coverage before interpreting curves; broad sweeps can leave a small, selected cohort. Fixed weights remain anchored to observed size, so individual fixed alternatives may lose local support at a distant supplied N even when the N-dependent mixture has support.

In [ ]:
display(result['support'].groupby(['analysis', 'hop'])[['n_cases', 'n_supported']].sum().assign(
    fraction=lambda t: t.n_supported / t.n_cases
))
fig = replacement_plot_support(result)
fig.savefig(OUTPUT_DIR / 'figures/01_support.png', dpi=160, bbox_inches='tight')
plt.show()

## Static contrasts at observed N

Rows identify the unchanged **recipient center**; columns identify the original **source cell** being replaced. Each panel compares the three methods within a fixed recipient/source identity and exact hop distance. Static effects are grouped into N bins: <150, 150–249, 250–349, 350–499, 500–799, ≥800; plotted x positions are the median observed counts of supported cases.

The two replacement curves overlap by construction here. A positive effect means the recipient's predicted curvature increases after replacement. The geometrically normalized effect is `ΔK / (4π / A_ref(N))`, with the existing area reference fitted on training organoids only. It is not a percentage change from the recipient's prediction. The per-organoid baseline cancels between intact and replaced predictions.

Cases and model seeds are averaged within organoid before organoids receive equal weight. Bands are pointwise organoid-bootstrap intervals conditional on fitted models and matching references; they do not include retraining or donor-estimation uncertainty. Panels have separate y scales.

In [ ]:
for hop in HOPS:
    fig = replacement_plot_pair_grid(result, analysis='observed', hop=hop,
                         metric=PLOT_METRIC, min_organoids=MIN_PLOT_ORGANOIDS)
    fig.savefig(OUTPUT_DIR / f'figures/02_static_hop{hop}_{PLOT_METRIC}.png', dpi=160, bbox_inches='tight')
    plt.show()

## Size sweeps on fixed neighborhoods

Only the supplied size changes in the FiLM and head; the graph and all intact fate vectors stay fixed. Fixed-reference curves isolate the model's response to size for an unchanged comparison distribution. N-dependent curves also include changes in the reference mixture. These are conditional network probes, not tracked developmental trajectories or physical fate conversions.

The normalized curves use `A_ref(supplied N)`. Change `PLOT_METRIC` to `delta_mu` to inspect curvature-unit contrasts without the geometric factor, or `delta_z` for the model's transformed output. Saved contrasts support these views without more inference.

N-dependent reference weights are local empirical estimates. Nearest-neighbor membership and support thresholds can produce changes in those weights; inspect the reference-mixture plot before attributing such bends to the neural network. `fixed_mass_supported_at_N` records how much of the frozen mixture has identity-level support in the N-dependent donor set at each supplied N.

In [ ]:
for hop in HOPS:
    fig = replacement_plot_pair_grid(result, analysis='sweep', hop=hop,
                         metric=PLOT_METRIC, min_organoids=MIN_PLOT_ORGANOIDS)
    fig.savefig(OUTPUT_DIR / f'figures/03_sweep_hop{hop}_{PLOT_METRIC}.png', dpi=160, bbox_inches='tight')
    plt.show()

## Checks and optional reference inspection

The following checks verify that static reference schemes coincide and that zeroing unassigned sources does nothing. The optional reference plot explains which replacements contribute to a selected curve. Every replacement contrast and its weights are retained in `result['cases']`; weights are `p_fixed_j` / `p_dependent_j`, and curvature contrasts are `delta_identity_j`, with `j` ordered as in `result['config']['markers']`.

A full vector replacement is always applied: the old identity is removed, then exactly one alternative marker is set (or all zeros for Unassigned). Unsupported alternatives can have an enumerated network prediction but receive no replacement probability. Do not interpret those individual predictions as supported biological alternatives.

In [ ]:
all_cases = result['cases']
static = all_cases[(all_cases.analysis == 'observed') & all_cases.fixed_supported]
np.testing.assert_allclose(static.replacement_fixed_delta_mu,
                           static.replacement_size_dependent_delta_mu)
unassigned = all_cases[all_cases.source_marker_name == 'Unassigned']
np.testing.assert_allclose(unassigned.zero_delta_mu, 0, atol=1e-10)
print('Static-mixture equality and unassigned no-op checks passed.')

# Choose identities after inspecting the all-pair figures; no pair is privileged.
REFERENCE_CENTER = None  # Recipient fate shown in the replacement-weight diagnostic.
REFERENCE_SOURCE = None  # Source fate shown in the replacement-weight diagnostic.
REFERENCE_HOP = 1  # Exact-hop ring shown in the replacement-weight diagnostic.
if REFERENCE_CENTER is not None and REFERENCE_SOURCE is not None:
    fig = replacement_plot_reference_weights(result, center=REFERENCE_CENTER,
                                  source=REFERENCE_SOURCE, hop=REFERENCE_HOP)
    plt.show()

display(result['summary'].head(12))

## Learned missing-fate ablation

Train and benchmark the models in **exclusive_fate_masking.ipynb** first. This extension compares all four interventions **within one masking-trained model**: zeroing, fixed-weight replacement, N-dependent replacement, and hiding fate with the trained flag. Recomputing the replacement contrasts on the new checkpoint avoids confusing an intervention change with a model change. Earlier figures retain the historical exclusive model and are labelled accordingly.

The exact old source/center cases and replacement weights are reused; nothing is resampled or rematched. The common support restriction and N-stable cohort are identical to the three-way comparison. The center remains observed and its prediction is compared with that same checkpoint's intact prediction. Zeroing unassigned is a no-op; hiding a known-unassigned identity generally is not.

This cell performs inference only. The default 2% masking rate is preselected, not optimized against the validation results.

In [ ]:
from src.analysis.interventions.masking_comparison import load_masking_comparison

# Saved model selection
MASKING_TRAINING_RUN = None  # Run name in training_results/fate_masking_training.
MASKING_RUN = resolve_training_run(ROOT, 'fate_masking_training', MASKING_TRAINING_RUN)

# Masking configuration
MASKING_RATE = .05  # Saved masking-training rate to evaluate.

# Paths and outputs
MASKING_OUTPUT_DIR = OUTPUT_DIR / f'masking_p{MASKING_RATE:g}'  # Masking comparison for this replacement run.

# Execution switches
RUN_MASKING_INFERENCE = False  # Generate intervention tables for masking-trained models.

if not (MASKING_RUN/'training_complete.json').exists():
    raise FileNotFoundError('Train exclusive_fate_masking.ipynb first, or set MASKING_RUN to a completed masking run.')
if RUN_MASKING_INFERENCE:
    run_masking_ablation(ROOT,OUTPUT_DIR,MASKING_RUN,MASKING_OUTPUT_DIR,
        rate=MASKING_RATE,batch_size=BATCH_SIZE,device=DEVICE)
mask_result=load_masking_comparison(MASKING_OUTPUT_DIR,bootstrap_samples=BOOTSTRAP_SAMPLES)
print(mask_result['model_label'])

### Static and swept four-way comparisons

All curves below come from the selected masking-trained checkpoint. Positive means higher predicted curvature after the intervention. Geometric normalization is identical to the earlier analysis. At observed N the two replacement mixtures still coincide; the unknown-fate reference can differ because it learns to predict from remaining information and does not exclude the source's original identity from what is plausible.

Historical-versus-new differences can reflect training augmentation, the new input channel, and the new checkpoint-selection protocol. Use the matched 0% control in the training notebook to assess augmentation effects.

In [ ]:
for analysis in ('observed','sweep'):
    for hop in HOPS:
        fig=replacement_plot_pair_grid(mask_result,analysis=analysis,hop=hop,
            metric=PLOT_METRIC,min_organoids=MIN_PLOT_ORGANOIDS)
        fig.savefig(MASKING_OUTPUT_DIR/f'figures/{analysis}_hop{hop}_{PLOT_METRIC}.png',dpi=160,bbox_inches='tight')
        plt.show()

# Explicit same-case comparison of the old and new checkpoint's zeroing effects.
old_zero=result['summary'].query("method == 'zero'")
new_zero=mask_result['summary'].query("method == 'zero'")
keys=['analysis','center_marker','source_marker_name','hop','coordinate','metric']
retraining_check=old_zero[keys+['mean','n_organoids']].merge(
    new_zero[keys+['mean','n_organoids']],on=keys,suffixes=('_historical','_mask_trained'),validate='one_to_one')
retraining_check['mean_change']=retraining_check['mean_mask_trained']-retraining_check['mean_historical']
retraining_check.to_csv(MASKING_OUTPUT_DIR/'historical_vs_mask_trained_zero.csv',index=False)
display(retraining_check.head(16))

## Optional: full versus exclusive fate encoding

Load or recompute the matched full/exclusive single-cell comparison. These are a separate pair of trained models and an explicitly matched cohort. Set the switch below only when examining that completed experiment; its settings and report remain attached to their source run.

In [ ]:
COMPARE_ENCODINGS = False  # Enable the optional full-versus-exclusive fate comparison.

# Full versus exclusive fate markers: size-conditioned FiLM ablation

This experiment uses only the **depth-two FiLM model with log cell count in FiLM and the head, and no global geometry in the head**. The exclusive model retains the original seven binary feature columns, with at most one positive entry. All-zero cells remain all zero. The original files and completed full-marker experiment are preserved.

Rules are applied in their supplied dictionary order: clear the named marker whenever any listed exclusion is *currently* positive, then continue to the next rule. Absent dataset markers are ignored. This is not the older export utility's rule evaluation against the original matrix.

For a controlled comparison, the original 1,100-organoid cohort, five folds, three seeds, baseline models, target transform, geometric calibration and N grid are reused. The 15 exclusive models are trained with the original hyperparameters and common-weight initialization. Full-marker checkpoints are reused and their validation predictions are rechecked against the saved results.

The default execution reads completed cached outputs. The switches below separately enable training, extended inference and summary recomputation.

In [ ]:
if COMPARE_ENCODINGS:
    from pathlib import Path
    import sys, json
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Markdown

    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/models/gnn.py').exists())  # Locate the repository from the notebook’s working directory.
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    # Saved model selection
    REFERENCE_RUN = PROJECT_ROOT / 'results_experiments/size_conditioned_ablation/20260910_164825_all_depth2_four_models_baseline_with_size'  # Saved reference run supplying cohort and preprocessing.

    # Paths and outputs
    COMPARISON_DIR = PROJECT_ROOT / 'results_experiments/size_conditioned_exclusive/20260914_ordered_exclusive_film_depth2'  # Directory holding matched full/exclusive comparison artifacts.

    # Saved model selection
    EXCLUSIVE_RUN = COMPARISON_DIR / 'exclusive'  # Saved run trained with exclusive fate markers.

    # Execution switches
    RUN_MAIN_ABLATIONS = False  # Recompute the matched full/exclusive ablation tables.
    RUN_EXTENDED_DIAGNOSTICS = False  # Recompute the optional niche and FiLM-route diagnostics.
    RECOMPUTE_SUMMARIES = False  # Rebuild summary tables from saved case-level results.
    from src.data.marker_exclusivity import EXCLUSIVITY_RULES
    print(json.dumps(EXCLUSIVITY_RULES, indent=2))
    settings = json.loads((EXCLUSIVE_RUN / 'settings.json').read_text())
    display(pd.Series({k: settings[k] for k in ['DATASET_NAME','MODEL_NAMES','MODEL_GLOBAL_FEATURES','NUM_LAYERS','HIDDEN_DIM',
        'FILM_HIDDEN_DIM','MODEL_SEEDS','MAX_EPOCHS','PATIENCE','LR','BATCH_SIZE','APPLY_SPHERICITY_FILTER','SPHERICITY_MAX',
        'SUBTRACT_CONSTANT_GLOBAL_BASELINE','BASELINE_FEATURES','BASELINE_REUSED']}, name='Experiment settings'))


In [ ]:
if COMPARE_ENCODINGS:
    from src.artifacts.provenance import workflow_fingerprint
    import copy, json, pickle, random, shutil
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    from torch_geometric.data import Data
    from src.data.marker_exclusivity import exclusive_markers, EXCLUSIVITY_RULES
    from src.data.io import load_graph_dataset_from_dir, select_graph_targets
    from src.data.metadata import attach_metadata_to_graphs, load_aux_metadata_for_dir, load_marker_names_from_dir
    from src.data.preprocessing import interpolate_target_outliers_from_neighbors
    from src.models.gnn import GINCurvature, SizeFiLMGINCurvature, count_parameters
    from src.training.loop import TrainConfig, train
    from src.training.losses import WeightedLossTerm, edge_loss_term
    from src.inference.predict import predict_targets
    from src.data.subgraphs import build_ego_subgraphs_for_dataset
    from src.analysis.interventions.size_sweeps import make_size_ablation_cases, evaluate_size_ablation
    from src.analysis.size_conditioning.cohort_inputs import seed_all, make_model, prepare_dataset, load_cohort, fold_graphs, mse_rows, full_fate_control, MODEL
    WORKFLOW_SOURCE = PROJECT_ROOT / 'experiments/ablation/fate_interventions.ipynb'  # Notebook used when recording the code fingerprint.


In [ ]:
if COMPARE_ENCODINGS:
    from src.artifacts import pickle_compat as artifact_pickle
    def evaluate_exclusive_models(root,reference,comparison,device='cuda'):
        root,reference,comparison=Path(root).resolve(),Path(reference).resolve(),Path(comparison).resolve()
        out=comparison/'exclusive';out.mkdir(parents=True,exist_ok=True)
        for folder in ['tables','checkpoints','figures','geometric_normalization']: (out/folder).mkdir(exist_ok=True)
        (comparison/'matched_full').mkdir(exist_ok=True)
        settings=json.loads((reference/'settings.json').read_text())
        settings=json.loads((out/'settings.json').read_text())
        dataset=root/'training_data'/settings['DATASET_NAME']
        markers=load_marker_names_from_dir(str(dataset))
        splits=json.loads((out/'splits.json').read_text());counts=json.loads((out/'sweep_grid.json').read_text())
        torch.set_num_threads(4)
        if device=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA unavailable')
        graphs=load_cohort(dataset,settings,reference)
        oldsettings=json.loads((reference/'settings.json').read_text())
        # Full x only is needed for matched inference; targets and edges are unchanged.
        full_x={}
        for g in graphs:
            with np.load(root/'training_data'/oldsettings['DATASET_NAME']/f'{g.organoid_str}.npz') as f:full_x[g.organoid_str]=torch.tensor(f['x'],dtype=torch.float32)
        for split in splits:
            fold=split['fold'];print(f'PREPARE fold={fold}',flush=True)
            with (out/f'checkpoints/fold_{fold}_preprocessing.pkl').open('rb') as f:pre=artifact_pickle.load(f)
            groups=fold_graphs(graphs,split,pre,settings)
            # Fixed random center sample, independently of fate. Exclusive-eligible
            # source choices are reused verbatim for the full model comparison.
            subgraphs=build_ego_subgraphs_for_dataset(groups['val'],num_hops=2,
                max_centers_per_graph=settings['ABLATION_CENTERS_PER_ORGANOID'],seed=settings['ABLATION_SEED']+fold)
            cases=make_size_ablation_cases(subgraphs,markers,hops=settings['ABLATION_HOPS'],seed=settings['ABLATION_SEED']+fold)
            indices=[];used={}
            for i,g in enumerate(subgraphs):
                if used.get(g.organoid_str,0)<settings['SWEEP_CENTERS_PER_ORGANOID']:indices.append(i)
                used[g.organoid_str]=used.get(g.organoid_str,0)+1
            remap={i:j for j,i in enumerate(indices)}
            sweeps=[dict(c,subgraph_index=remap[c['subgraph_index']]) for c in cases if c['subgraph_index'] in remap]
            for name,manifest in [('ablation',cases),('sweep',sweeps)]:
                (out/f'tables/fold_{fold}_{name}_manifest.json').write_text(json.dumps(manifest,indent=2))
            fullsubs=[]
            for sub in subgraphs:
                g=copy.copy(sub);g.x=full_x[g.organoid_str][g.orig_nodes].clone();fullsubs.append(g)
            for seed in settings['MODEL_SEEDS']:
                for encoding,folder,subs,modelrun in [('exclusive',out/'tables',subgraphs,out),('full_matched',comparison/'matched_full',fullsubs,reference)]:
                    model=make_model(settings,markers)
                    model.load_state_dict(torch.load(modelrun/f'checkpoints/fold_{fold}_seed_{seed}_{MODEL}.pt',map_location='cpu',weights_only=True))
                    kwargs=dict(model=model,target_transform=pre['residual_transform'],size_center=pre['size_center'],size_scale=pre['size_scale'],
                        batch_size=settings['PERTURB_BATCH_SIZE'],device=device,max_fold_change=settings['LOCAL_SIZE_FOLD_CHANGE'])
                    for n in ['observed',*counts]:
                        filename=f'fold_{fold}_seed_{seed}_{MODEL}_'+('observed' if n=='observed' else f'N{n}_sweep')+'.csv'
                        path=folder/filename
                        if path.exists():continue
                        result=evaluate_size_ablation(subs if n=='observed' else [subs[i] for i in indices],
                            cases if n=='observed' else sweeps,count=None if n=='observed' else n,**kwargs)
                        result.assign(fold=fold,seed=seed,model=MODEL,encoding=encoding).to_csv(path,index=False)
                        print(f'ABLATION {encoding} fold={fold} seed={seed} N={n}',flush=True)
            # Reproduce the original full-model validation with reused preprocessing.
            fullval=[]
            for g in groups['val']:
                view=copy.copy(g);view.x=full_x[g.organoid_str];fullval.append(view)
            oldmse=pd.read_csv(reference/'tables/validation_organoid_mse.csv')
            for seed in settings['MODEL_SEEDS']:
                model=make_model(settings,markers)
                model.load_state_dict(torch.load(reference/f'checkpoints/fold_{fold}_seed_{seed}_{MODEL}.pt',map_location='cpu',weights_only=True))
                check=pd.DataFrame(mse_rows(model,fullval,pre['residual_transform'],fold,seed,device))
                match=check.merge(oldmse[(oldmse.fold==fold)&(oldmse.seed==seed)&(oldmse.model==MODEL)],on='organoid_str',suffixes=('_new','_old'))
                np.testing.assert_allclose(match.mse_new,match.mse_old,atol=2e-8,rtol=2e-4)
                check.to_csv(comparison/'matched_full'/f'fold_{fold}_seed_{seed}_validation.csv',index=False)
            print(f'FOLD COMPLETE {fold}',flush=True)
        pd.concat([pd.read_csv(p) for p in sorted((out/'tables').glob('*_validation.csv'))]).to_csv(out/'tables/validation_organoid_mse.csv',index=False)
        (out/'complete.json').write_text(json.dumps(dict(checkpoints=len(splits)*len(settings['MODEL_SEEDS']),full_validation_reproduced=True)))
        return out

In [ ]:
# Study-specific functions are defined in this notebook.
if COMPARE_ENCODINGS:
    if RUN_MAIN_ABLATIONS:
        evaluate_exclusive_models(PROJECT_ROOT, REFERENCE_RUN, COMPARISON_DIR, device='cuda')
    if RUN_EXTENDED_DIAGNOSTICS:
        from src.analysis.size_conditioning.cohort_inputs import full_fate_control
        from src.analysis.size_conditioning.encoding_effects import run_diagnostics
        full_fate_control(PROJECT_ROOT, REFERENCE_RUN, COMPARISON_DIR, device='cuda')
        run_diagnostics(COMPARISON_DIR, PROJECT_ROOT, device='cuda', niche_summary=niche_statistics_summarize)
        from src.analysis.size_conditioning.pair_control import run as run_pair_control
        run_pair_control(COMPARISON_DIR, PROJECT_ROOT, device='cuda')
    if RECOMPUTE_SUMMARIES:
        from src.analysis.size_conditioning.encoding_effects import summarize, summarize_diagnostic_comparison
        summarize(COMPARISON_DIR)
        summarize_diagnostic_comparison(COMPARISON_DIR)
        from src.analysis.size_conditioning.encoding_effects import summarize_pair_control
        summarize_pair_control(COMPARISON_DIR)

    assert (EXCLUSIVE_RUN / 'complete.json').exists(), 'Main experiment is incomplete.'
    assert (COMPARISON_DIR / 'diagnostics_complete.json').exists(), 'Extended diagnostics are incomplete.'
    assert (COMPARISON_DIR / 'comparison_complete.json').exists(), 'Comparison summaries are incomplete.'
    # Study-specific functions are defined in this notebook.
    comparison_figures = encoding_save_figures(COMPARISON_DIR)
    print('Results:', COMPARISON_DIR)


### Encoding audit and validation

Keeping the same feature width and model architecture makes the comparison more controlled. An exclusive cell's vector is a one-hot representation; unmarked cells retain the original all-zero representation, without adding a new class. Validation MSE is in physical curvature squared, with equal organoid weighting and seeds averaged within organoids. Target residualization is unchanged.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['01_encoding_audit'])
    plt.close(comparison_figures['01_encoding_audit'])
    display(comparison_figures['02_validation_mse'])
    plt.close(comparison_figures['02_validation_mse'])

### Primary LGR5-recipient ablation comparison

**Blue and orange compare identical exclusive-eligible recipients and source cells.** The full model receives their original full marker vectors throughout the graph; the exclusive model receives the exclusive vectors. Recipient identities are held fixed within each ablation. Matching labels use the recipient's retained exclusive marker, so blue curves are not re-expanded across its original coexpressed markers.

The gray dotted curves reproduce the original full-marker sampling cohorts. A gray-to-blue difference reflects case eligibility, sampling and labeling changes; a blue-to-orange difference reflects the new representation and retrained model on matched cells.

The y-axis is `ΔK × Â_train-fold(N)/(4π)`, where `ΔK = after source-marker removal − intact prediction`. Negative means removal lowers predicted curvature. Observed-size curves group different organoids by actual cell count; sweeps keep each graph and case fixed while changing the FiLM and head size inputs. They are not longitudinal trajectories.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['04_lgr5_sweep_comparison'])
    plt.close(comparison_figures['04_lgr5_sweep_comparison'])
    display(comparison_figures['05_observed_and_sweep'])
    plt.close(comparison_figures['05_observed_and_sweep'])

### Control for how much source fate is removed

Exclusive one-marker removal leaves the source vector all zero. Original one-marker removal may leave other positive markers behind. The green control therefore removes *all* marker bits from the same source in the original full-marker model. It preserves the recipient and all other cells. This helps assess perturbation severity; the intact graph representations and fitted models still differ.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['05b_full_source_fate_control'])
    plt.close(comparison_figures['05b_full_source_fate_control'])

### All marker pairs: size sweeps

Rows denote unchanged recipient markers, including unmarked recipients. Columns denote source markers removed at the indicated exact hop. Curves with fewer than five contributing organoids are masked. Every eligible exclusive case is paired with a full-marker evaluation on the same cells.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['03_sweep_all_markers_hop1'])
    plt.close(comparison_figures['03_sweep_all_markers_hop1'])
    display(comparison_figures['03_sweep_all_markers_hop2'])
    plt.close(comparison_figures['03_sweep_all_markers_hop2'])

### All marker pairs: observed sizes

These curves use observed N, grouped into the same original cohort quintiles. The original full-marker curves can have different recipient/source cohorts. Confidence intervals condition on fitted models and calibration and do not include retraining uncertainty.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['03_observed_all_markers_hop1'])
    plt.close(comparison_figures['03_observed_all_markers_hop1'])
    display(comparison_figures['03_observed_all_markers_hop2'])
    plt.close(comparison_figures['03_observed_all_markers_hop2'])

### Relative source preference around LGR5 recipients

For the same LGR5-positive recipient and hop, average absolute single-source effects within candidate source groups A and B. Their relative sensitivity is `D = (S_A − S_B)/(S_A + S_B)`. Positive favors A; negative favors B. This is not an additive share of the prediction or total abundance-weighted influence. The positive geometric factor cancels in D.

Agr2 and Chroma represent exclusive-eligible precursor-marker states; Lysozyme and Serotonin represent their retained mature markers. The full model's paired results use these same selected cells but retain their full original features. Double-positive *encoded* states disappear by construction; this does not imply biological coexpression or intermediate states are absent.

Gray curves retain the previous native-cohort result for context. Matched blue and orange comparisons are the primary encoding comparison. Transformed-target curves check dependence on the nonlinear inverse transform.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['06_niche_dominance'])
    plt.close(comparison_figures['06_niche_dominance'])
    display(comparison_figures['07_dominance_output_scale_check'])
    plt.close(comparison_figures['07_dominance_output_scale_check'])
    display(comparison_figures['08_observed_niche_dominance'])
    plt.close(comparison_figures['08_observed_niche_dominance'])

### Redundancy, recipient specificity and detected crypt context

Lysozyme–Serotonin factorial contrasts remove markers from two distinct source cells around an unchanged LGR5 recipient: `both removed − Lysozyme removed − Serotonin removed + intact`. A negative interaction with negative individual effects is consistent with buffering in the model; it does not identify a biological signaling mechanism.

Backup-context matching uses the same source, organoid, hop, encoded recipient marker signature and crypt region, plus coarsened degree and local LGR5 fraction. The paired full evaluation uses these identical cases. Context labels here derive from the exclusive sampling representation; original additional marker bits remain visible to the full model.

Crypt labels describe observed, sufficiently mature structures. No detected crypt is not a known pre-crypt state, and the destructive snapshots cannot establish formation-to-maintenance transitions.

The green factorial control removes **all positive marker bits from each selected source cell** in the original full-marker model, using the same recipients and source pairs. This matches the all-zero source state produced by exclusive-marker removal and tests whether differences in interaction magnitude arise from the amount of fate information removed.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['09_redundancy_comparison'])
    plt.close(comparison_figures['09_redundancy_comparison'])
    display(comparison_figures['10_backup_context'])
    plt.close(comparison_figures['10_backup_context'])
    display(comparison_figures['11_specificity'])
    plt.close(comparison_figures['11_specificity'])
    display(comparison_figures['11_crypt_paired'])
    plt.close(comparison_figures['11_crypt_paired'])

### FiLM routes and representation changes

The conditioning routes independently sweep both FiLM and head inputs, FiLM alone, head alone, or one FiLM layer, anchored at N=361. These interventions need not add linearly.

Hidden cosine similarities compare each model's own ablation response to its own direction at N=361. Hidden axes are not aligned between independently trained models. The common-gain diagnostic asks whether a marker-pair response vector across N can be explained by a shared positive multiplier of its N=361 vector; the unexplained norm fraction is not an explained-variance statistic.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['12_conditioning_routes'])
    plt.close(comparison_figures['12_conditioning_routes'])
    display(comparison_figures['13_hidden_direction'])
    plt.close(comparison_figures['13_hidden_direction'])
    display(comparison_figures['14_common_scaling_check'])
    plt.close(comparison_figures['14_common_scaling_check'])

### Secondary center-removal diagnostic

This supplementary factorial experiment removes center LGR5 as well as a neighboring marker. It changes recipient identity and is therefore more delicate than the primary experiments above. In the exclusive model it erases the center's entire encoded fate, while the original full model may retain other center markers. Interpret these comparisons accordingly.

In [ ]:
if COMPARE_ENCODINGS:
    display(comparison_figures['15_center_removal_secondary'])
    plt.close(comparison_figures['15_center_removal_secondary'])

### Findings and validation

The report below summarizes the completed comparison. Case-level data, matched support, paired endpoint contrasts and seed summaries are saved in the result directory.

In [ ]:
if COMPARE_ENCODINGS:
    display(Markdown((COMPARISON_DIR / 'REPORT.md').read_text()))
    display(pd.Series(json.loads((COMPARISON_DIR / 'validation.json').read_text()), name='Validation checks'))